In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_linear_schedule_with_warmup
)
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

# Kelas Konfigurasi untuk menampung semua hyperparameter
class CFG:
    # Path Data
    COMP_PATH = "/kaggle/input/llm-classification-finetuning"
    
    # Model & Tokenizer
    MODEL_NAME = "microsoft/deberta-v3-small"
    MAX_LENGTH = 400  
    
    # Training
    EPOCHS = 2
    BATCH_SIZE = 16  
    LEARNING_RATE = 1e-5
    
    # Lain-lain
    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_LABELS = 3  # (A, B, Tie)

# Fungsi untuk seeding agar hasil reproducible
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
print(f"Menggunakan device: {CFG.DEVICE}")

In [ ]:
train = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
test = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')
submission = pd.read_csv('/kaggle/input/llm-classification-finetuning/sample_submission.csv')

In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train Data:")
train.info()
print("\nTest Data:")
test.info()

In [ ]:
print("\nFirst 5 rows of Train Data:")
train.head()

In [ ]:
print("\nStatistik deskriptif data:")
train.describe().T

In [ ]:
print("\nNilai yang hilang :")
train.isnull().sum()

In [ ]:
# Distribusi Target
target_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
target_counts = train[target_cols].sum()

plt.figure(figsize=(10, 6))
sns.barplot(x=target_counts.index, y=target_counts.values, palette="viridis")
plt.title('Distribusi Pemenang (Target)')
plt.ylabel('Jumlah')
plt.xlabel('Pemenang')
plt.show()

In [ ]:
# Analisis Panjang Teks
train['prompt_len'] = train['prompt'].apply(len)
train['response_a_len'] = train['response_a'].apply(len)
train['response_b_len'] = train['response_b'].apply(len)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(train['prompt_len'], ax=axes[0], color='skyblue', kde=True)
axes[0].set_title('Distribusi Panjang Prompt')
sns.histplot(train['response_a_len'], ax=axes[1], color='salmon', kde=True)
axes[1].set_title('Distribusi Panjang Respons A')
sns.histplot(train['response_b_len'], ax=axes[2], color='lightgreen', kde=True)
axes[2].set_title('Distribusi Panjang Respons B')
plt.tight_layout()
plt.show()

In [ ]:
# Word Cloud untuk Prompt
prompt_text = ' '.join(train['prompt'].dropna())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(prompt_text)

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud untuk Prompt')
plt.show()

del prompt_text, wordcloud
gc.collect()

In [ ]:
# Fungsi untuk membuat input teks tunggal
def create_text_input(df):
    # Format [CLS] PROMPT [SEP] RESP A [SEP] RESP B [SEP]
    return "PROMPT: " + df["prompt"] + \
           " [SEP] RESPONSE A: " + df["response_a"] + \
           " [SEP] RESPONSE B: " + df["response_b"]

# Terapkan fungsi ke data training dan test
train['text'] = create_text_input(train)
test['text'] = create_text_input(test)

# Buat kolom target tunggal (0: A, 1: B, 2: Tie)
conditions = [
    train['winner_model_a'] == 1,
    train['winner_model_b'] == 1,
    train['winner_tie'] == 1
]
choices = [0, 1, 2]
train['target'] = np.select(conditions, choices)

# Inisialisasi tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

# Bagi data menjadi training dan validasi
train_data, val_data = train_test_split(
    train,
    test_size=0.1,
    random_state=CFG.SEED,
    stratify=train['target']
)

print(f"Ukuran data training: {len(train_data)}")
print(f"Ukuran data validasi: {len(val_data)}")

# Membuat PyTorch Dataset
class PreferenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = df['text'].values
        self.is_test = is_test
        if not self.is_test:
            self.labels = df['target'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text,
            truncation=True,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        item = {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }
        
        if not self.is_test:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
            
        return item

In [ ]:
# Memuat model
model = AutoModelForSequenceClassification.from_pretrained(
    CFG.MODEL_NAME,
    num_labels=CFG.NUM_LABELS
)
model.to(CFG.DEVICE)

# Membuat dataset dan dataloader
train_dataset = PreferenceDataset(train_data, tokenizer, CFG.MAX_LENGTH)
val_dataset = PreferenceDataset(val_data, tokenizer, CFG.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=2)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE)
num_training_steps = len(train_loader) * CFG.EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0.1 * num_training_steps,
    num_training_steps=num_training_steps
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Loop Training
best_val_logloss = float('inf')
best_model_path = "best_model.pth"

for epoch in range(CFG.EPOCHS):
    print(f"======== Epoch {epoch + 1}/{CFG.EPOCHS} ========")
    
    # --- Training ---
    model.train()
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc="Training")
    
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(CFG.DEVICE)
        attention_mask = batch['attention_mask'].to(CFG.DEVICE)
        labels = batch['labels'].to(CFG.DEVICE)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        progress_bar.set_postfix({'loss': loss.item()})
        
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Average Training Loss: {avg_train_loss:.4f}")

    # --- Evaluasi ---
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch['input_ids'].to(CFG.DEVICE)
            attention_mask = batch['attention_mask'].to(CFG.DEVICE)
            labels = batch['labels'].to(CFG.DEVICE)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            all_preds.append(logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    # Hitung LogLoss untuk validasi
    preds_np = np.concatenate(all_preds)
    labels_np = np.concatenate(all_labels)
    
    # Softmax untuk mendapatkan probabilitas
    probs_np = torch.softmax(torch.tensor(preds_np), dim=1).numpy()
    
    val_logloss = log_loss(labels_np, probs_np)
    print(f"Validation LogLoss: {val_logloss:.4f}")
    
    if val_logloss < best_val_logloss:
        best_val_logloss = val_logloss
        torch.save(model.state_dict(), best_model_path)
        print(f"Model terbaik disimpan di '{best_model_path}' dengan LogLoss: {best_val_logloss:.4f}")

    gc.collect()

In [ ]:
# Muat state dict dari model terbaik yang disimpan
model.load_state_dict(torch.load(best_model_path))
model.to(CFG.DEVICE)
print("Model terbaik dimuat untuk evaluasi akhir.")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Final Evaluation"):
        input_ids = batch['input_ids'].to(CFG.DEVICE)
        attention_mask = batch['attention_mask'].to(CFG.DEVICE)
        labels = batch['labels'].to(CFG.DEVICE)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        all_preds.append(logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

preds_np = np.concatenate(all_preds)
labels_np = np.concatenate(all_labels)
probs_np = torch.softmax(torch.tensor(preds_np), dim=1).numpy()

final_val_logloss = log_loss(labels_np, probs_np)

print(f"\nSkor LogLoss Akhir pada Set Validasi: {final_val_logloss:.4f}")

In [ ]:
# Buat dataset dan dataloader untuk test set
test_dataset = PreferenceDataset(test, tokenizer, CFG.MAX_LENGTH, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=2)

# Fungsi prediksi
def predict(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            input_ids = batch['input_ids'].to(CFG.DEVICE)
            attention_mask = batch['attention_mask'].to(CFG.DEVICE)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            
            # Terapkan softmax untuk mendapatkan probabilitas
            probs = torch.softmax(outputs.logits, dim=1)
            all_preds.append(probs.cpu().numpy())
            
    return np.concatenate(all_preds)

# Dapatkan prediksi
predictions = predict(model, test_loader)

print("\nBentuk array prediksi:", predictions.shape)
print("Contoh 5 probabilitas prediksi pertama:")
print(predictions[:5])

In [ ]:
# Isi dataframe submission dengan hasil prediksi
submission['winner_model_a'] = predictions[:, 0]
submission['winner_model_b'] = predictions[:, 1]
submission['winner_tie'] = predictions[:, 2]

# Simpan ke file csv
submission.to_csv("submission.csv", index=False)

print("\nFile submission.csv berhasil dibuat!")
display(submission.head())